# Bird Species Audio Classification — Google Colab Multi-Model GPU Pipeline

This notebook trains, benchmarks, and ensembles **three distinct Deep Learning architectures** on a Google Colab GPU (`T4` / `L4` / `A100`):
1. **EfficientNet-B0** (2D CNN)
2. **CRNN** (ResNet-34 + Bidirectional GRU + Temporal Attention Pooling)
3. **AST** (Audio Spectrogram Transformer, MIT AudioSet pretrained)
4. **3-Model Probability Ensemble**

Supports **16 Iconic Indian Birds (`indian16`)**, **Auto Top-N Indian Birds (`auto_india_top`, e.g. 50 or 100+ species)**, **Core 8 (`core8`)**, or the full **Kaggle BirdCLEF+ 2025 (`kaggle`)** benchmark — plus **BirdNET v2.4 (6,500+ species)** & an **Evaluation Benchmarks tab** in the Gradio app.

## 1. Check GPU & Install System Dependencies

In [ ]:
import torch
print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU for fast training.")
!nvidia-smi || true
!apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1

## 2. Clone Repository, Install Dependencies & Verify Unit Tests

In [ ]:
import os
import subprocess
from pathlib import Path

def run_cmd(cmd, env=None):
    merged_env = {**os.environ, "PYTHONUNBUFFERED": "1", "TF_CPP_MIN_LOG_LEVEL": "3", **(env or {})}
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=merged_env
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)

REPO_URL = "https://github.com/Haricharanv/bird-audio-classification.git"
REPO_DIR = Path("/content/bird-audio-classification")

if not REPO_DIR.exists():
    run_cmd(["git", "clone", REPO_URL, str(REPO_DIR)])
else:
    run_cmd(["git", "-C", str(REPO_DIR), "pull"])
os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())

run_cmd(["pip", "install", "-q", "-r", "requirements.txt"])
run_cmd(["pytest", "tests/", "-v"])

## 3. Optional: Mount Google Drive (Persist Trained Models)

In [ ]:
USE_GOOGLE_DRIVE = False  # Set True to save model bundles directly to Google Drive
DRIVE_SAVE_DIR = Path("/content/drive/MyDrive/bird_audio_runs")

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_SAVE_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Bundles will be copied to: {DRIVE_SAVE_DIR}")

## 4. Download Bird Dataset (`"indian16"`, `"auto_india_top"`, `"core8"`, or `"kaggle"`)
- **`"indian16"`**: 16 iconic Indian & common birds (Indian Peafowl, Asian Koel, Common Myna, House Crow, Rose-ringed Parakeet, Coppersmith Barbet, House Sparrow, etc.) — 40 clips each (640 real recordings).
- **`"auto_india_top"`**: Automatically queries iNaturalist for the top `AUTO_INDIA_TOP_N` (e.g. `50` or `100`) most-recorded Indian birds with research-grade audio.
- **`"core8"`**: 8 global birds (40 clips each).
- **`"kaggle"`**: Full 206-species BirdCLEF+ 2025 dataset (requires `kaggle.json`).

In [ ]:
DATASET_MODE = "indian16"  # "indian16" | "auto_india_top" | "core8" | "kaggle"
AUTO_INDIA_TOP_N = 50      # Used when DATASET_MODE == "auto_india_top" (try 50 or 100)
INAT_PER_SPECIES = 40

if DATASET_MODE in ("indian16", "core8"):
    RAW_DIR = Path(f"/content/data/{DATASET_MODE}")
    CACHE_DIR = Path(f"/content/cache_{DATASET_MODE}")
    META_DIR = Path(f"data/metadata_{DATASET_MODE}")
    NOISE_PATH = Path(f"data/noise/noise_bank_{DATASET_MODE}.npy")
    run_cmd([
        "python", "scripts/download_inat_birds.py",
        "--out_dir", str(RAW_DIR),
        "--per_species", str(INAT_PER_SPECIES),
        "--preset", DATASET_MODE,
        "--workers", "8",
    ])
elif DATASET_MODE == "auto_india_top":
    RAW_DIR = Path(f"/content/data/india_top_{AUTO_INDIA_TOP_N}")
    CACHE_DIR = Path(f"/content/cache_india_top_{AUTO_INDIA_TOP_N}")
    META_DIR = Path(f"data/metadata_india_top_{AUTO_INDIA_TOP_N}")
    NOISE_PATH = Path(f"data/noise/noise_bank_india_top_{AUTO_INDIA_TOP_N}.npy")
    run_cmd([
        "python", "scripts/download_inat_birds.py",
        "--out_dir", str(RAW_DIR),
        "--per_species", str(INAT_PER_SPECIES),
        "--auto_india_top", str(AUTO_INDIA_TOP_N),
        "--workers", "8",
    ])
elif DATASET_MODE == "kaggle":
    from google.colab import files
    RAW_DIR = Path("/content/data/birdclef-2025")
    CACHE_DIR = Path("/content/cache_birdclef")
    META_DIR = Path("data/metadata_birdclef")
    NOISE_PATH = Path("data/noise/noise_bank_birdclef.npy")
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(exist_ok=True)
    if not (kaggle_dir / "kaggle.json").is_file():
        print("Upload your kaggle.json token file:")
        uploaded = files.upload()
        for _, data in uploaded.items():
            (kaggle_dir / "kaggle.json").write_bytes(data)
    os.chmod(kaggle_dir / "kaggle.json", 0o600)
    run_cmd(["pip", "install", "-q", "kaggle"])
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    run_cmd(["kaggle", "competitions", "download", "-c", "birdclef-2025", "-p", str(RAW_DIR)])
    run_cmd(["unzip", "-q", "-o", str(RAW_DIR / "birdclef-2025.zip"), "-d", str(RAW_DIR)])
else:
    raise ValueError("DATASET_MODE must be 'indian16', 'auto_india_top', 'core8', or 'kaggle'")

## 5. Prepare Zero-Leakage Splits, Noise Bank, and Cached Training Chunks

In [ ]:
run_cmd([
    "python", "scripts/prepare_data.py",
    "--config", "configs/effnet_b0.yaml",
    "debug=false",
    f"data.raw_dir={RAW_DIR}",
    f"data.cache_dir={CACHE_DIR}",
    f"data.metadata_dir={META_DIR}",
    f"data.noise_bank_path={NOISE_PATH}",
    "data.min_files_per_species=10",
    "data.max_chunks_per_species=200",
    "data.split_by=filename",
    "audio.fmin=150.0",
])

## 6. Train Multiple Architectures (`EfficientNet-B0`, `CRNN`, `AST`)
Edit `MODELS_TO_TRAIN` if you want to run a subset (e.g., omit `AST` if short on time).

In [ ]:
import glob
import shutil

MODELS_TO_TRAIN = [
    ("EfficientNet-B0", "configs/effnet_b0.yaml", 25, 32),
    ("CRNN (ResNet34+BiGRU)", "configs/crnn.yaml", 25, 32),
    ("AST Transformer", "configs/ast.yaml", 15, 16),
]

has_gpu = torch.cuda.is_available()
DEVICE = "cuda" if has_gpu else "cpu"
WORKERS = 2 if has_gpu else 0
PIN_MEM = "true" if has_gpu else "false"

trained_runs = []  # List of (label, cfg_path, run_dir)

for label, cfg_path, epochs, bs in MODELS_TO_TRAIN:
    print("\n" + "=" * 70)
    print(f"TRAINING: {label} ({cfg_path})")
    print("=" * 70)
    before = set(glob.glob("runs/*/best_model.pt"))
    mp = "fp16" if has_gpu else "no"
    cmd = [
        "python", "scripts/train.py",
        "--config", cfg_path,
        "debug=false",
        f"system.device={DEVICE}",
        f"system.mixed_precision={mp}",
        f"system.num_workers={WORKERS}",
        f"system.pin_memory={PIN_MEM}",
        f"data.raw_dir={RAW_DIR}",
        f"data.cache_dir={CACHE_DIR}",
        f"data.metadata_dir={META_DIR}",
        f"data.noise_bank_path={NOISE_PATH}",
        "audio.fmin=150.0",
        f"train.epochs={epochs if has_gpu else 10}",
        f"train.batch_size={bs if has_gpu else 16}",
    ]
    if not has_gpu:
        cmd.extend(["augment.waveform.pitch_shift_prob=0.0", "augment.waveform.time_stretch_prob=0.0"])
    run_cmd(cmd)

    after = set(glob.glob("runs/*/best_model.pt"))
    new_ckpts = sorted(after - before, key=os.path.getmtime, reverse=True)
    ckpt_path = Path(new_ckpts[0]).resolve() if new_ckpts else Path(sorted(after, key=os.path.getmtime, reverse=True)[0]).resolve()
    tax_src = RAW_DIR / "taxonomy.csv"
    if tax_src.is_file():
        shutil.copy2(tax_src, ckpt_path.parent / "taxonomy.csv")
    trained_runs.append((label, cfg_path, ckpt_path))

print("\nTrained checkpoints:")
for label, _, ckpt_path in trained_runs:
    print(f"  - {label}: {ckpt_path}")

## 7. Evaluate Individual Models + 3-Model Ensemble on Held-Out Test Split

In [ ]:
import json
import numpy as np
import pandas as pd
from IPython.display import Image, display
from src.data.dataset import BirdDataset
from src.evaluation.metrics import compute_classification_metrics
from src.models.ensemble import load_ensemble_from_checkpoints
from src.utils.audio_inference import aggregate_window_probs
from src.utils.config import load_config

comparison_rows = []

for label, cfg_path, ckpt_path in trained_runs:
    run_cmd([
        "python", "scripts/evaluate.py",
        "--checkpoint", str(ckpt_path),
        "--config", cfg_path,
        "--split", "test",
        f"data.raw_dir={RAW_DIR}",
        f"data.cache_dir={CACHE_DIR}",
        f"data.metadata_dir={META_DIR}",
        "audio.fmin=150.0",
    ])
    m = json.loads((ckpt_path.parent / "evaluation_test" / "metrics.json").read_text())
    comparison_rows.append({
        "Model": label,
        "Macro-F1": round(m["macro_f1"], 4),
        "Top-1 Acc": f"{m['top1_accuracy']:.2%}",
        "Top-3 Acc": f"{m['top3_accuracy']:.2%}",
        "ROC-AUC": round(m["roc_auc"], 4),
    })

if len(trained_runs) >= 2:
    ref_cfg = load_config(trained_runs[0][1], [f"data.raw_dir={RAW_DIR}", f"data.metadata_dir={META_DIR}", "audio.fmin=150.0"])
    splits_df = pd.read_csv(META_DIR / "splits.csv", dtype={"primary_label": str, "filename": str})
    test_df = splits_df[splits_df["split"] == "test"].reset_index(drop=True)
    first_ckpt = torch.load(trained_runs[0][2], map_location="cpu")
    class_to_idx = first_ckpt["class_to_idx"]
    test_ds = BirdDataset(df=test_df, cfg=ref_cfg, split="test", species_to_idx=class_to_idx, use_sliding_windows=True)

    ens_model = load_ensemble_from_checkpoints([p for _, _, p in trained_runs], ref_cfg, device=DEVICE)
    y_true, y_probs = [], []
    for idx in range(len(test_ds)):
        sample = test_ds[idx]
        wf = sample["waveform"].to(DEVICE)
        with torch.no_grad():
            probs = ens_model(wf).cpu().numpy()
        y_probs.append(aggregate_window_probs(probs, windows=sample["waveform"].numpy(), mode="confidence_energy"))
        y_true.append(sample["target"].numpy())
    ens_res = compute_classification_metrics(np.array(y_true), np.stack(y_probs, axis=0))
    comparison_rows.append({
        "Model": f"Ensemble ({len(trained_runs)} models)",
        "Macro-F1": round(ens_res["macro_f1"], 4),
        "Top-1 Acc": f"{ens_res['top1_accuracy']:.2%}",
        "Top-3 Acc": f"{ens_res['top3_accuracy']:.2%}",
        "ROC-AUC": round(ens_res["roc_auc"], 4),
    })

comp_df = pd.DataFrame(comparison_rows)
print("\n" + "=" * 70)
print("HELD-OUT TEST SPLIT MODEL COMPARISON")
print("=" * 70)
print(comp_df.to_string(index=False))

best_eval_dir = trained_runs[0][2].parent / "evaluation_test"
for img_name in ["confusion_matrix.png", "sample_gradcam.png"]:
    img_path = best_eval_dir / img_name
    if img_path.is_file():
        display(Image(filename=str(img_path)))

## 8. Export All Trained Models & Demo Clips for Local App
Unzip the exported archive inside `bird-audio-classification/` locally to load all 3 models + Ensemble in `app/app.py`.

In [ ]:
import zipfile

splits_df = pd.read_csv(META_DIR / "splits.csv", dtype={"primary_label": str, "filename": str})
test_df = splits_df[splits_df["split"] == "test"]
demo_dir = RAW_DIR / "demo_examples"
demo_dir.mkdir(parents=True, exist_ok=True)
for sp, grp in test_df.groupby("primary_label"):
    src_clip = RAW_DIR / "train_audio" / grp.iloc[0]["filename"]
    if src_clip.is_file():
        shutil.copy2(src_clip, demo_dir / f"{sp}_test.ogg")

bundle_path = Path(f"/content/{DATASET_MODE}_models_bundle.zip")
with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for _, _, ckpt_path in trained_runs:
        rdir = ckpt_path.parent
        for f in rdir.rglob("*"):
            if f.is_file() and f.name != "last_model.pt":
                zf.write(f, arcname=f"runs/{rdir.name}/{f.relative_to(rdir)}")
    for ex in demo_dir.glob("*.ogg"):
        zf.write(ex, arcname=f"demo_examples/{ex.name}")
    if (RAW_DIR / "taxonomy.csv").is_file():
        zf.write(RAW_DIR / "taxonomy.csv", arcname="taxonomy.csv")

print(f"Created export archive: {bundle_path} ({bundle_path.stat().st_size / 1e6:.1f} MB)")
if USE_GOOGLE_DRIVE:
    shutil.copy2(bundle_path, DRIVE_SAVE_DIR / bundle_path.name)
    print(f"Copied archive to Google Drive: {DRIVE_SAVE_DIR / bundle_path.name}")
else:
    from google.colab import files
    files.download(str(bundle_path))

## 9. Launch Interactive Gradio Demo (Custom Models + Ensemble + BirdNET v2.4 + Benchmarks Tab)

In [ ]:
import app.app as bird_app
bird_app.MODEL_CACHE.clear()
bird_app.create_demo_audio_examples = lambda _dir: [[str(p)] for p in sorted(demo_dir.glob("*.ogg"))]
demo = bird_app.build_app()
demo.launch(share=True, debug=False, allowed_paths=[str(RAW_DIR), str(REPO_DIR / "runs")])